# ADS FA2 - Case Study: Data Analysis Using Machine Learning & Streamlit

| | |
|---|---|
| **Department** | MCA |
| **Year** | SYMCA |
| **Semester** | 1 |
| **Course** | Advanced Data Science (MCA33PE17) |
| **Student** | 125M1H045 |
| **Assignment** | FA2 — Data Analysis Case Study |

---

## Dataset: UCI Energy Efficiency Dataset

**Source:** UCI Machine Learning Repository  
**Records:** 768 buildings | **Features:** 8 | **Targets:** 2 (Heating Load, Cooling Load)

### Feature Description

| Feature | Name | Description |
|---|---|---|
| X1 | Relative Compactness | Ratio of volume to surface area |
| X2 | Surface Area (m²) | Total building surface area |
| X3 | Wall Area (m²) | Total wall area |
| X4 | Roof Area (m²) | Total roof area |
| X5 | Overall Height (m) | Building height |
| X6 | Orientation | Direction (2=N, 3=E, 4=S, 5=W) |
| X7 | Glazing Area | Window area as % of floor |
| X8 | Glazing Distribution | How windows are distributed |
| Y1 | Heating Load (kWh/m²) | **Target for Regression** |
| Y2 | Cooling Load (kWh/m²) | **Target for Classification** |

### ML Tasks

| # | Algorithm | Prediction Task | Type |
|---|---|---|---|
| 1 | Linear Regression | Predict exact **Heating Load** (kWh/m²) | Regression |
| 2 | Random Forest | Predict **High or Low** Heating Load | Binary Classification |
| 3 | K-Nearest Neighbors | Predict **Cooling Load Class** (Low/Medium/High) | Multi-class Classification |
| 4 | K-Means Clustering | Group buildings by **energy efficiency profile** | Unsupervised Clustering |

---
## Part 1: Data Acquisition & Preprocessing

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

### 1.1 Load Dataset

In [ ]:
# Load Energy Efficiency dataset
df = pd.read_excel('dataset/energy_efficiency.xlsx')

# Rename columns with meaningful names
df.columns = ['Compactness', 'SurfaceArea', 'WallArea', 'RoofArea',
              'Height', 'Orientation', 'GlazingArea', 'GlazingDist',
              'HeatingLoad', 'CoolingLoad']

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset info
print('Dataset Info:')
df.info()

In [ ]:
# Basic statistics
print('Basic Statistics:')
df.describe().round(2)

### 1.2 Exploratory Data Analysis (EDA)

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Distribution of target variables
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['HeatingLoad'], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Heating Load (Y1)')
axes[0].set_xlabel('Heating Load (kWh/m²)')
axes[0].set_ylabel('Count')

axes[1].hist(df['CoolingLoad'], bins=20, color='coral', edgecolor='black')
axes[1].set_title('Distribution of Cooling Load (Y2)')
axes[1].set_xlabel('Cooling Load (kWh/m²)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap - Energy Efficiency Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Key features vs Heating Load
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].scatter(df['Compactness'], df['HeatingLoad'], alpha=0.5, color='steelblue', s=20)
axes[0,0].set_xlabel('Compactness'); axes[0,0].set_ylabel('Heating Load')
axes[0,0].set_title('Compactness vs Heating Load')

axes[0,1].scatter(df['SurfaceArea'], df['HeatingLoad'], alpha=0.5, color='coral', s=20)
axes[0,1].set_xlabel('Surface Area'); axes[0,1].set_ylabel('Heating Load')
axes[0,1].set_title('Surface Area vs Heating Load')

axes[1,0].scatter(df['Height'], df['HeatingLoad'], alpha=0.5, color='green', s=20)
axes[1,0].set_xlabel('Height'); axes[1,0].set_ylabel('Heating Load')
axes[1,0].set_title('Height vs Heating Load')

axes[1,1].scatter(df['GlazingArea'], df['HeatingLoad'], alpha=0.5, color='purple', s=20)
axes[1,1].set_xlabel('Glazing Area'); axes[1,1].set_ylabel('Heating Load')
axes[1,1].set_title('Glazing Area vs Heating Load')

plt.tight_layout()
plt.show()

In [ ]:
# Heating Load vs Cooling Load
plt.figure(figsize=(8, 5))
plt.scatter(df['HeatingLoad'], df['CoolingLoad'], alpha=0.5, color='orange', s=20)
plt.title('Heating Load vs Cooling Load')
plt.xlabel('Heating Load (kWh/m²)')
plt.ylabel('Cooling Load (kWh/m²)')
plt.tight_layout()
plt.show()

### 1.3 Data Preprocessing

In [ ]:
# Drop missing values
df = df.dropna()
print('After cleaning - Shape:', df.shape)

FEATURES = ['Compactness', 'SurfaceArea', 'WallArea', 'RoofArea',
            'Height', 'Orientation', 'GlazingArea', 'GlazingDist']

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])
joblib.dump(scaler, 'scaler/scaler.pkl')
print('Features scaled and scaler saved!')

In [ ]:
# Create target variables

# For Model 1 (Linear Regression) - continuous Heating Load
y_reg = df['HeatingLoad'].values

# For Model 2 (Random Forest) - High/Low Heating Load
median_hl = df['HeatingLoad'].median()
df['hl_class'] = (df['HeatingLoad'] >= median_hl).astype(int)
y_clf = df['hl_class'].values
print('Median Heating Load:', round(median_hl, 2), 'kWh/m2')

# For Model 3 (KNN) - Cooling Load classes (Low/Medium/High)
q33 = df['CoolingLoad'].quantile(0.33)
q66 = df['CoolingLoad'].quantile(0.66)
df['cl_class'] = df['CoolingLoad'].apply(
    lambda x: 0 if x <= q33 else (1 if x <= q66 else 2))
y_knn = df['cl_class'].values
print('Cooling Load Thresholds - Low:', round(q33,2), '| Med:', round(q66,2))

# Train/Test Splits
X_tr_reg,  X_te_reg,  y_tr_reg,  y_te_reg  = train_test_split(X, y_reg,  test_size=0.2, random_state=42)
X_tr_clf,  X_te_clf,  y_tr_clf,  y_te_clf  = train_test_split(X, y_clf,  test_size=0.2, random_state=42)
X_tr_knn,  X_te_knn,  y_tr_knn,  y_te_knn  = train_test_split(X, y_knn,  test_size=0.2, random_state=42)

print('\nTrain size:', X_tr_reg.shape[0], '| Test size:', X_te_reg.shape[0])

---
## Part 2: Model Implementation & Evaluation

### Model 1: Linear Regression — Predict Heating Load

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train model
lr_model = LinearRegression()
lr_model.fit(X_tr_reg, y_tr_reg)
y_pred_lr = lr_model.predict(X_te_reg)

# Metrics
train_r2 = r2_score(y_tr_reg, lr_model.predict(X_tr_reg))
test_r2  = r2_score(y_te_reg, y_pred_lr)
mae      = mean_absolute_error(y_te_reg, y_pred_lr)
rmse     = np.sqrt(mean_squared_error(y_te_reg, y_pred_lr))

print('=== Linear Regression Results ===')
print('Train R2 Score:', round(train_r2, 4), '-- Train Accuracy:', str(round(train_r2*100, 2))+'%')
print('Test  R2 Score:', round(test_r2, 4),  '-- Test  Accuracy:', str(round(test_r2*100, 2))+'%')
print('MAE  :', round(mae, 4), 'kWh/m2')
print('RMSE :', round(rmse, 4), 'kWh/m2')

joblib.dump(lr_model, 'models/linear_regression.pkl')
print('Model saved!')

In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(8, 5))
plt.scatter(y_te_reg, y_pred_lr, alpha=0.6, color='steelblue', s=30)
plt.plot([y_te_reg.min(), y_te_reg.max()], [y_te_reg.min(), y_te_reg.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.title('Linear Regression: Actual vs Predicted Heating Load')
plt.xlabel('Actual Heating Load (kWh/m²)')
plt.ylabel('Predicted Heating Load (kWh/m²)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Coefficients for Linear Regression
coef_df = pd.Series(lr_model.coef_, index=FEATURES).sort_values(key=abs, ascending=False)
plt.figure(figsize=(8, 5))
coef_df.plot(kind='bar', color=['steelblue' if c > 0 else 'coral' for c in coef_df], edgecolor='black')
plt.title('Linear Regression - Feature Coefficients')
plt.xlabel('Feature')
plt.ylabel('Coefficient Value')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Model 2: Random Forest — High/Low Heating Load

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

# Train with GridSearchCV
print('Running GridSearchCV...')
param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 15, None]}
gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_tr_clf, y_tr_clf)

rf_model  = gs.best_estimator_
y_pred_rf = rf_model.predict(X_te_clf)
train_acc = accuracy_score(y_tr_clf, rf_model.predict(X_tr_clf))
test_acc  = accuracy_score(y_te_clf, y_pred_rf)

print('=== Random Forest Results ===')
print('Best Params :', gs.best_params_)
print('Train Accuracy:', str(round(train_acc*100, 2))+'%')
print('Test  Accuracy:', str(round(test_acc*100, 2))+'%')
print('Precision :', round(precision_score(y_te_clf, y_pred_rf), 4))
print('Recall    :', round(recall_score(y_te_clf, y_pred_rf), 4))
print('F1 Score  :', round(f1_score(y_te_clf, y_pred_rf), 4))
print()
print(classification_report(y_te_clf, y_pred_rf, target_names=['Low Load', 'High Load']))

joblib.dump(rf_model, 'models/random_forest.pkl')
joblib.dump(median_hl, 'models/median_hl.pkl')
print('Model saved!')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_te_clf, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Load', 'High Load'],
            yticklabels=['Low Load', 'High Load'])
plt.title('Random Forest - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Random Forest - Feature Importance')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Model 3: KNN — Cooling Load Class (Low / Medium / High)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Find best K
k_values   = range(1, 21)
accuracies = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_knn, y_tr_knn)
    accuracies.append(accuracy_score(y_te_knn, knn.predict(X_te_knn)))

best_k = list(k_values)[accuracies.index(max(accuracies))]
print('Best K:', best_k, '| Best Test Accuracy:', str(round(max(accuracies)*100, 2))+'%')

# Plot K vs Accuracy
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), accuracies, marker='o', color='purple')
plt.axvline(x=best_k, color='red', linestyle='--', label='Best K='+str(best_k))
plt.title('KNN: K Value vs Test Accuracy')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Train KNN with best K
knn_model  = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_tr_knn, y_tr_knn)
y_pred_knn = knn_model.predict(X_te_knn)
train_knn  = accuracy_score(y_tr_knn, knn_model.predict(X_tr_knn))
test_knn   = accuracy_score(y_te_knn, y_pred_knn)

print('=== KNN Results ===')
print('Train Accuracy:', str(round(train_knn*100, 2))+'%')
print('Test  Accuracy:', str(round(test_knn*100, 2))+'%')
print('Precision:', round(precision_score(y_te_knn, y_pred_knn, average='weighted'), 4))
print('Recall   :', round(recall_score(y_te_knn, y_pred_knn, average='weighted'), 4))
print('F1 Score :', round(f1_score(y_te_knn, y_pred_knn, average='weighted'), 4))
print()
print(classification_report(y_te_knn, y_pred_knn, target_names=['Low', 'Medium', 'High']))

joblib.dump(knn_model, 'models/knn_model.pkl')
joblib.dump({'q33': q33, 'q66': q66}, 'models/knn_thresholds.pkl')
print('Model saved!')

In [ ]:
# KNN Confusion Matrix
cm_knn = confusion_matrix(y_te_knn, y_pred_knn)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.title('KNN - Confusion Matrix (Cooling Load Class)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### Model 4: K-Means Clustering — Building Energy Profiles

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Elbow Method
inertias = []
K_range  = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertias, marker='o', color='orange')
plt.axvline(x=4, color='red', linestyle='--', label='Chosen K=4')
plt.title('Elbow Method — Optimal Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Train K-Means with K=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
km4.fit(X)
df['cluster'] = km4.labels_

CLUSTER_LABELS = {0: 'High Efficiency', 1: 'Medium Efficiency',
                  2: 'Low Efficiency',  3: 'Very Low Efficiency'}
df['cluster_name'] = df['cluster'].map(CLUSTER_LABELS)

# Cluster distribution
plt.figure(figsize=(7, 5))
df['cluster_name'].value_counts().plot(kind='bar',
    color=['green','blue','orange','red'], edgecolor='black')
plt.title('Building Clusters - Energy Efficiency Groups')
plt.xlabel('Cluster')
plt.ylabel('Number of Buildings')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

joblib.dump(km4, 'models/kmeans_model.pkl')
print('K-Means model saved!')

In [ ]:
# Visualize Clusters using PCA (2D)
pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X)

colors = ['green', 'blue', 'orange', 'red']
plt.figure(figsize=(8, 6))
for c_id, c_name in CLUSTER_LABELS.items():
    mask = df['cluster'] == c_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                color=colors[c_id], label=c_name, alpha=0.6, s=30)
plt.title('K-Means Clusters — Building Energy Profiles (PCA 2D)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend()
plt.tight_layout()
plt.show()

---
## Model Comparison Summary

In [ ]:
# Summary of all models
summary = {
    'Model': ['Linear Regression', 'Random Forest', 'KNN', 'K-Means'],
    'Task': [
        'Predict Heating Load (kWh/m²)',
        'High / Low Heating Load',
        'Cooling Load Class (Low/Med/High)',
        'Building Energy Cluster'
    ],
    'Type': ['Regression', 'Classification', 'Classification', 'Clustering'],
    'Train Score': [
        str(round(r2_score(y_tr_reg, lr_model.predict(X_tr_reg))*100, 2))+'%',
        str(round(accuracy_score(y_tr_clf, rf_model.predict(X_tr_clf))*100, 2))+'%',
        str(round(accuracy_score(y_tr_knn, knn_model.predict(X_tr_knn))*100, 2))+'%',
        'N/A (Unsupervised)'
    ],
    'Test Score': [
        str(round(r2_score(y_te_reg, y_pred_lr)*100, 2))+'% R2',
        str(round(accuracy_score(y_te_clf, y_pred_rf)*100, 2))+'% Acc',
        str(round(accuracy_score(y_te_knn, y_pred_knn)*100, 2))+'% Acc',
        'K=4 Clusters'
    ]
}
print('=== Final Model Comparison ===')
pd.DataFrame(summary)

---
## All models saved! Run the Streamlit app:
```bash
streamlit run app.py
```